# Setup

In [1]:
SPARK_START_FROM_SCRATCH = True
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = ["10.15.30.1"]
SPARK_VPN_DOMAIN = "rgorosti.vpn.itam.mx"

SPARK_DOCKER_BASE = "spark:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JUPYTER_LAB_DOCKER_TAG = "spark-jupyter:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JOB_VENV_DOCKER_TAG = "spark-job-venv:3.5.7-scala2.12-java17-python3-ubuntu"
SPARK_JOB_VENV_BUILD_DIR = "/opt/spark/venv-build"

SPARK_MASTER_NAME = "spark-master"
SPARK_MASTER_HOSTNAME = f"{SPARK_MASTER_NAME}.{SPARK_VPN_DOMAIN}"
# SPARK_MASTER_IP = "10.15.20.2"
SPARK_MASTER_WUBUI_PORT = 6080
SPARK_MASTER_PORT = 6077

SPARK_TOTAL_WORKERS = 3
SPARK_WORKER_NAMES = [f"spark-worker-{i+1}" for i in range(SPARK_TOTAL_WORKERS)]
SPARK_WORKER_HOSTNAMES = [
    f"{SPARK_WORKER_NAMES[i]}.{SPARK_VPN_DOMAIN}" for i in range(SPARK_TOTAL_WORKERS)
]
SPARK_WORKER_IPS = ["10.15.30.18"] * SPARK_TOTAL_WORKERS
SPARK_WORKER_WEBUI_PORTS = [6080 + (i + 1) for i in range(SPARK_TOTAL_WORKERS)]

SPARK_WORKDIR = "/opt/spark/work-dir"

JUPYTER_LAB_NAME = "spark-jupyter"
JUPYTER_LAB_HOSTNAME = f"{JUPYTER_LAB_NAME}.{SPARK_VPN_DOMAIN}"
# JUPYTER_LAB_IP = "10.15.20.2"
JUPYTER_LAB_PORT = 6888
JUPYTER_LAB_MONITOR_PORT = 4040
JUPYTER_LAB_TOKEN = ""

SPARK_SHARED_WORKSPACE = "shared-workspace"
SPARK_SHARED_WORKSPACE_DIR = f"/opt/spark/{SPARK_SHARED_WORKSPACE}"

In [2]:
import os
from pathlib import Path

LOCALHOST_WORKDIR = f"{os.path.join(os.path.relpath(Path.cwd()))}"
DOCKER_MOUNTDIR = os.path.join(LOCALHOST_WORKDIR, "mount")

path = Path(LOCALHOST_WORKDIR)
path.mkdir(parents=True, exist_ok=True)

# Stop spark-cluster.docker-compose.yml

In [3]:
!docker compose -f spark-cluster.docker-compose.yml down -v

In [4]:
import shutil

if SPARK_START_FROM_SCRATCH :
    # shutil.rmtree(os.path.join(DOCKER_MOUNTDIR, SPARK_SHARED_WORKSPACE, "spark-warehouse"), ignore_errors=True)
    # shutil.rmtree(
    #     os.path.join(DOCKER_MOUNTDIR, SPARK_SHARED_WORKSPACE, "iceberg-warehouse"), ignore_errors=True
    # )
    shutil.rmtree(os.path.join(DOCKER_MOUNTDIR, SPARK_MASTER_NAME), ignore_errors=True)
    for spark_worker_name in SPARK_WORKER_NAMES:
        shutil.rmtree(
            os.path.join(DOCKER_MOUNTDIR, spark_worker_name), ignore_errors=True
        )
    shutil.rmtree(os.path.join(DOCKER_MOUNTDIR, SPARK_MASTER_NAME), ignore_errors=True)
    shutil.rmtree(os.path.join(DOCKER_MOUNTDIR, SPARK_SHARED_WORKSPACE), ignore_errors=True)
    
    Path(os.path.join(DOCKER_MOUNTDIR, JUPYTER_LAB_NAME)).mkdir(parents=True, exist_ok=True)

### Build spark-jupyter

In [5]:
import os
from IPython.display import Markdown, display

dockerfile_spark_jupyter_python_packages = (
    "pyspark==3.5.7 delta-spark==3.2.0 jupyterlab pandas pyarrow"
)

dockerfile_spark_jupyter_name = "dockerfile.spark-jupyter"

# language=dockerfile
dockerfile_spark_jupyter_contents = f""" 

# Use the official Spark image as the base
FROM apache/{SPARK_DOCKER_BASE}

# Switch to root to install software
USER root

# Set the working directory
WORKDIR {SPARK_WORKDIR}

# Expose the Jupyter port
EXPOSE 8888

# Install Python dependencies
RUN apt-get update && apt-get install -y python3-venv
RUN python3 -m pip install --no-cache-dir {dockerfile_spark_jupyter_python_packages}

# Set the default command to launch Jupyter Lab
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--NotebookApp.token=$$JUPYTER_LAB_TOKEN"]
"""

with open(
    os.path.join(LOCALHOST_WORKDIR, dockerfile_spark_jupyter_name), "w"
) as spark_compose_yaml_file:
    spark_compose_yaml_file.write(dockerfile_spark_jupyter_contents.strip())

print(
    f"Successfully created: '{os.path.relpath(os.path.join(LOCALHOST_WORKDIR,dockerfile_spark_jupyter_name))}'"
)
display(Markdown(f"```dockerfile\n{dockerfile_spark_jupyter_contents}\n```"))

Successfully created: 'dockerfile.spark-jupyter'


```dockerfile
 

# Use the official Spark image as the base
FROM apache/spark:3.5.7-scala2.12-java17-python3-ubuntu

# Switch to root to install software
USER root

# Set the working directory
WORKDIR /opt/spark/work-dir

# Expose the Jupyter port
EXPOSE 8888

# Install Python dependencies
RUN apt-get update && apt-get install -y python3-venv
RUN python3 -m pip install --no-cache-dir pyspark==3.5.7 delta-spark==3.2.0 jupyterlab pandas pyarrow

# Set the default command to launch Jupyter Lab
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--NotebookApp.token=$$JUPYTER_LAB_TOKEN"]

```

In [6]:
!docker build -t {SPARK_JUPYTER_LAB_DOCKER_TAG} -f dockerfile.spark-jupyter .



[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.1s (2/2)                                          docker:default
 => [internal] load build definition from dockerfile.spark-jupyter         0.0s
 => => transferring dockerfile: 660B                                       0.0s
 => [internal] load metadata for docker.io/apache/spark:3.5.7-scala2.12-j  0.1s
[+] Building 0.2s (7/8)                                          docker:default
 => [internal] load build definition from dockerfile.spark-jupyter         0.0s
 => => transferring dockerfile: 660B                                       0.0s
 => [internal] load metadata for docker.io/apache/spark:3.5.7-scala2.12-j  0.1s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
 => [1/4] FROM docker.io/apache/spark:3.5.7-scala2.12-java17-python3-ubun  0.0s
 => => resolve docker.io/apache/spark:

### Build spark-job-venv

In [7]:
import os
from IPython.display import Markdown, display

dockerfile_spark_job_venv_name = "dockerfile.spark-job-venv"
dockerfile_spark_job_venv_contents = f"""
# Use the previously generated spark-jupyter image as the base
FROM {SPARK_JUPYTER_LAB_DOCKER_TAG}

# Create virtual env for spark jobs
RUN mkdir -p {SPARK_JOB_VENV_BUILD_DIR} && \\
        cd {SPARK_JOB_VENV_BUILD_DIR} && \\
        python3 -m venv --copies spark_job_env && \\
        {SPARK_JOB_VENV_BUILD_DIR}/spark_job_env/bin/pip install venv-pack pandas pyarrow faker faker-commerce mimesis && \\
        {SPARK_JOB_VENV_BUILD_DIR}/spark_job_env/bin/venv-pack -p spark_job_env -o spark_job_env.tar.gz
"""

with open(os.path.join(LOCALHOST_WORKDIR, dockerfile_spark_job_venv_name), "w") as spark_compose_yaml_file:
    spark_compose_yaml_file.write(dockerfile_spark_job_venv_contents.strip())

print(f"Successfully created: '{os.path.relpath(os.path.join(LOCALHOST_WORKDIR,dockerfile_spark_jupyter_name))}'")
display(Markdown(f"```dockerfile\n{dockerfile_spark_job_venv_contents}\n```"))

Successfully created: 'dockerfile.spark-jupyter'


```dockerfile

# Use the previously generated spark-jupyter image as the base
FROM spark-jupyter:3.5.7-scala2.12-java17-python3-ubuntu

# Create virtual env for spark jobs
RUN mkdir -p /opt/spark/venv-build && \
        cd /opt/spark/venv-build && \
        python3 -m venv --copies spark_job_env && \
        /opt/spark/venv-build/spark_job_env/bin/pip install venv-pack pandas pyarrow faker faker-commerce mimesis && \
        /opt/spark/venv-build/spark_job_env/bin/venv-pack -p spark_job_env -o spark_job_env.tar.gz

```

In [8]:
from pathlib import Path
path = Path(DOCKER_MOUNTDIR)
path.mkdir(parents=True, exist_ok=True)

!docker build -t {SPARK_JOB_VENV_DOCKER_TAG} -f dockerfile.spark-job-venv .
!docker create --name spark-job-venv {SPARK_JOB_VENV_DOCKER_TAG}
!docker cp spark-job-venv:{SPARK_JOB_VENV_BUILD_DIR}/spark_job_env.tar.gz "{DOCKER_MOUNTDIR}/{JUPYTER_LAB_NAME}/spark_job_env.tar.gz"
!docker rm spark-job-venv



[+] Building 0.0s (0/1)                                          docker:default
 => [internal] load build definition from dockerfile.spark-job-venv        0.0s
[+] Building 0.1s (2/3)                                          docker:default
 => [internal] load build definition from dockerfile.spark-job-venv        0.0s
 => => transferring dockerfile: 558B                                       0.0s
 => [internal] load metadata for docker.io/library/spark-jupyter:3.5.7-sc  0.1s
[+] Building 0.2s (3/5)                                          docker:default
 => [internal] load build definition from dockerfile.spark-job-venv        0.0s
 => => transferring dockerfile: 558B                                       0.0s
 => [internal] load metadata for docker.io/library/spark-jupyter:3.5.7-sc  0.1s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
 => [1/2] FROM docker.io/library/spark

# Start spark.docker-compose.yml

In [9]:
import os
import yaml
from IPython.display import Markdown, display

SPARK_VSCODE_SERVER_DIR = os.path.join(LOCALHOST_WORKDIR, "vscode_server")
SPARK_MOUNT_JARS = [
    f"{os.path.join(LOCALHOST_WORKDIR,"jars",file)}:/opt/spark/jars/{file}"
    for file in os.listdir(os.path.join(LOCALHOST_WORKDIR, "jars"))
    if file.endswith("jar")
]

spark_compose_dict = {
    "name": "spark-cluster",
    "networks": {"spark-cluster": {"driver": "bridge"}},
    "services": {
        SPARK_MASTER_NAME: {
            "image": f"apache/{SPARK_DOCKER_BASE}",
            "container_name": SPARK_MASTER_NAME,
            "user": "root",
            "command": f'bash -c "/opt/spark/bin/spark-class org.apache.spark.deploy.$$SPARK_MODE.$${{SPARK_MODE^}} --host {SPARK_MASTER_HOSTNAME} --port $$SPARK_MASTER_PORT --webui-port $$SPARK_MASTER_WEBUI_PORT"',
            "environment": [
                "PYSPARK_PYTHON=python3",
                "SPARK_MODE=master",
                f"SPARK_MASTER_PORT={SPARK_MASTER_PORT}",
                f"SPARK_MASTER_WEBUI_PORT={SPARK_MASTER_WUBUI_PORT}",
                "SPARK_DAEMON_MEMORY=1G",
            ],
            "volumes": [
                f"{os.path.join(DOCKER_MOUNTDIR,SPARK_SHARED_WORKSPACE)}:{SPARK_SHARED_WORKSPACE_DIR}",
                f"{os.path.join(DOCKER_MOUNTDIR,SPARK_MASTER_NAME)}:{SPARK_WORKDIR}"
            ]
            + SPARK_MOUNT_JARS,
            "networks": ["spark-cluster"],
            "hostname": SPARK_MASTER_HOSTNAME,
            "ports": [
                f"{SPARK_MASTER_WUBUI_PORT}:{SPARK_MASTER_WUBUI_PORT}",
                f"{SPARK_MASTER_PORT}:{SPARK_MASTER_PORT}",
            ],
            "extra_hosts": [
                f"{DOCKER_INTERNAL_HOST}:host-gateway",
            ],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "2.0", "memory": "1G"}}},
            "healthcheck": {
                "test": [
                    "CMD",
                    "curl",
                    "-f",
                    f"http://{SPARK_MASTER_HOSTNAME}:{SPARK_MASTER_WUBUI_PORT}",
                ],
                "interval": "10s",
                "timeout": "10s",
                "retries": 10,
                "start_period": "10s",
            },
        },
        "spark-jupyter": {
            "image": SPARK_JUPYTER_LAB_DOCKER_TAG,
            "container_name": "spark-jupyter",
            "user": "root",
            "command": [
                "bash",
                "-c",
                " ".join(
                    [
                        "jupyter lab",
                        "--ip=0.0.0.0",
                        f"--port={JUPYTER_LAB_PORT}",
                        "--no-browser",
                        "--allow-root",
                        f"--NotebookApp.token='{JUPYTER_LAB_TOKEN}'",
                        "--NotebookApp.password=''",
                        "--NotebookApp.allow_origin='*'",
                        "--ServerApp.disable_check_xsrf=True",
                        f"--ServerApp.root_dir={SPARK_WORKDIR}",
                    ]
                ),
            ],
            "environment": [
                "PYSPARK_PYTHON=python3",
                # f"JUPYTER_LAB_PORT={JUPYTER_LAB_PORT}",
                # f"JUPYTER_LAB_TOKEN={JUPYTER_LAB_TOKEN}",
                "SPARK_EXECUTOR_MEMORY=1536M",
            ],
            "volumes": [
                f"{os.path.join(DOCKER_MOUNTDIR,SPARK_SHARED_WORKSPACE)}:{SPARK_SHARED_WORKSPACE_DIR}",
                f"{os.path.join(DOCKER_MOUNTDIR,JUPYTER_LAB_NAME)}:{SPARK_WORKDIR}",
                f"{SPARK_VSCODE_SERVER_DIR}:/root/.vscode-server",
            ]
            + SPARK_MOUNT_JARS,
            "networks": ["spark-cluster"],
            "hostname": JUPYTER_LAB_HOSTNAME,
            "ports": [
                f"{JUPYTER_LAB_PORT}:{JUPYTER_LAB_PORT}",
                f"{JUPYTER_LAB_MONITOR_PORT}:{JUPYTER_LAB_MONITOR_PORT}",
            ],
            "extra_hosts": [
                f"{DOCKER_INTERNAL_HOST}:host-gateway",
            ],
            "dns": DOCKER_DNS,
            "deploy": {"resources": {"limits": {"cpus": "2.0", "memory": "1G"}}},
            "healthcheck": {
                "test": [
                    "CMD",
                    "curl",
                    "-f",
                    f"http://{JUPYTER_LAB_HOSTNAME}:{JUPYTER_LAB_PORT}",
                ],
                "interval": "10s",
                "timeout": "10s",
                "retries": 10,
                "start_period": "10s",
            },
        },
    },
}

for i in range(SPARK_TOTAL_WORKERS):

    spark_compose_dict["services"][SPARK_WORKER_NAMES[i]] = {
        "image": f"apache/{SPARK_DOCKER_BASE}",
        "container_name": SPARK_WORKER_NAMES[i],
        "user": "root",
        "command": f'bash -c "/opt/spark/bin/spark-class org.apache.spark.deploy.$$SPARK_MODE.$${{SPARK_MODE^}} $$SPARK_MASTER_URL --host {SPARK_WORKER_HOSTNAMES[i]} --webui-port $$SPARK_WORKER_WEBUI_PORT"',
        "environment": [
            "PYSPARK_PYTHON=python3",
            "SPARK_MODE=worker",
            "SPARK_WORKER_CORES=2",
            "SPARK_DAEMON_MEMORY=512M",
            "SPARK_WORKER_MEMORY=2048M",
            f"SPARK_WORKER_WEBUI_PORT={SPARK_WORKER_WEBUI_PORTS[i]}",
            f"SPARK_MASTER_URL=spark://{SPARK_MASTER_HOSTNAME}:{SPARK_MASTER_PORT}",
        ],
        "volumes": [
            f"{os.path.join(DOCKER_MOUNTDIR,SPARK_SHARED_WORKSPACE)}:{SPARK_SHARED_WORKSPACE_DIR}",
            f"{os.path.join(DOCKER_MOUNTDIR,SPARK_WORKER_NAMES[i])}:{SPARK_WORKDIR}"
        ]
        + SPARK_MOUNT_JARS,
        "networks": ["spark-cluster"],
        "hostname": SPARK_WORKER_HOSTNAMES[i],
        "ports": [f"{SPARK_WORKER_WEBUI_PORTS[i]}:{SPARK_WORKER_WEBUI_PORTS[i]}"],
        "extra_hosts": [
            f"{DOCKER_INTERNAL_HOST}:host-gateway",
        ],
        "dns": DOCKER_DNS,
        "deploy": {"resources": {"limits": {"cpus": "4.0", "memory": "1.5G"}}},
        "depends_on": {
            "spark-master": {"condition": "service_healthy"},
            "spark-jupyter": {"condition": "service_healthy"},
        }
        | {
            SPARK_WORKER_NAMES[j]: {"condition": "service_started"} for j in range(0, i)
        },
        "healthcheck": {
            "test": [
                "CMD",
                "curl",
                "-f",
                f"http://{SPARK_WORKER_HOSTNAMES[i]}:{SPARK_WORKER_WEBUI_PORTS[i]}",
            ],
            "interval": "10s",
            "timeout": "10s",
            "retries": 10,
            "start_period": "10s",
        },
    }

# 3. Dump the dictionary to a YAML file
spark_compose_yaml_path = os.path.join(
    LOCALHOST_WORKDIR, "spark-cluster.docker-compose.yml"
)
spark_compose_yaml_contents = yaml.dump(
    spark_compose_dict, default_flow_style=False, sort_keys=False, indent=4
)
with open(spark_compose_yaml_path, "w") as spark_compose_yaml_file:
    spark_compose_yaml_file.write(spark_compose_yaml_contents)

print(f"Successfully created: '{os.path.relpath(spark_compose_yaml_path)}'")
display(Markdown(f"```yaml\n{spark_compose_yaml_contents}\n```"))

FileNotFoundError: [Errno 2] No such file or directory: './jars'

In [ ]:
!docker compose -f spark-cluster.docker-compose.yml up -d --wait

 Network spark-cluster_spark-cluster Creating 
 Network spark-cluster_spark-cluster Created 
 Container spark-jupyter Creating 
 Container spark-master Creating 
 Container spark-jupyter Created 
 Container spark-master Created 
 Container spark-worker-1 Creating 
 Container spark-worker-1 Created 
 Container spark-worker-2 Creating 
 Container spark-worker-2 Created 
 Container spark-worker-3 Creating 
 Container spark-worker-3 Created 
 Container spark-jupyter Starting 
 Container spark-master Starting 
 Container spark-jupyter Started 
 Container spark-master Started 
 Container spark-jupyter Waiting 
 Container spark-master Waiting 
 Container spark-jupyter Healthy 
 Container spark-master Healthy 
 Container spark-worker-1 Starting 
 Container spark-worker-1 Started 
 Container spark-master Waiting 
 Container spark-jupyter Waiting 
 Container spark-jupyter Healthy 
 Container spark-master Healthy 
 Container spark-worker-2 Starting 
 Container spark-worker-2 Started 
 Container s